In [40]:
import polars as pl
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

In [41]:
class Pipeline():

    def __init__(self):
        self.scaler = None
        self.pca = None
        self.zipcode_data = None

    """
    Training Pipeline
    """
    def make_zipcode_metadata(self, X_train: pl.DataFrame):
        self.zipcode_data = (X_train
                            .group_by("zipcode")
                            .agg(
                                median_price = pl.col("price").median(),
                                median_condition = pl.col("condition").median(),
                                median_grade = pl.col("grade").median()
                                ))
        X_train = X_train.join(self.zipcode_data, on="zipcode", how="inner")
        
        return X_train, self.zipcode_data

    def fit_pca(self, X_train: pl.DataFrame, 
                cols=["bedrooms", "bathrooms", "sqft_living", "sqft_lot", "floors", "sqft_above", "sqft_basement"]):
        
        self.scaler = StandardScaler()
        self.pca = PCA(3)

        pca_cols = X_train.select(cols).to_numpy()
        pca_cols_scaled = self.scaler.fit_transform(pca_cols)
        components = self.pca.fit_transform(pca_cols_scaled)
        components_df = pl.DataFrame(schema=["PC1", "PC2", "PC3"], data=components)
        X_train = X_train.drop(cols).hstack(components_df)

        return X_train, components
    
    def drop_cols(self, X: pl.DataFrame, cols=["date", "id", "yr_renovated", "zipcode"]):
        return X.drop(cols)
    
    def run_train_preprocessing(self, X_train):
        X_train = self.make_zipcode_metadata(X_train)[0]
        X_train = self.fit_pca(X_train)[0]
        X_train = self.drop_cols(X_train)

        return X_train
    

    """
    Inference Pipeline
    """
    def enrich(self, X_test: pl.DataFrame
           ) -> pl.DataFrame:
        
        enriched_df = X_test.join(self.zipcode_data, on="zipcode", how="inner")
        
        return enriched_df
    
    def run_pca(self, X_test: pl.DataFrame,
                cols=["bedrooms", "bathrooms", "sqft_living", "sqft_lot", "floors", "sqft_above", "sqft_basement"]):

        pca_cols = X_test.select(cols).to_numpy()
        pca_cols_scaled = self.scaler.transform(pca_cols)
        components = self.pca.transform(pca_cols_scaled)
        components_df = pl.DataFrame(schema=["PC1", "PC2", "PC3"], data=components)
        X_test = X_test.drop(cols).hstack(components_df)
        return X_test

    def run_inference_pipeline(self, X_test):
        X_test = self.enrich(X_test)
        X_test = self.run_pca(X_test)
        X_test = self.drop_cols(X_test)

        return X_test

In [42]:
df = pl.read_csv("data/raw.csv")

In [43]:
pipeline = Pipeline()

X = pipeline.run_train_preprocessing(df)
X.head(3)

waterfront,view,condition,grade,yr_built,lat,long,sqft_living15,sqft_lot15,price,median_price,median_condition,median_grade,PC1,PC2,PC3
i64,i64,i64,i64,i64,f64,f64,i64,i64,f64,f64,f64,f64,f64,f64,f64
0,0,3,8,2007,47.3862,-122.048,3280,4033,429900.0,341000.0,3.0,7.0,2.977914,0.850386,-0.574438
0,0,2,7,1979,47.3035,-122.382,1310,7865,233000.0,269000.0,3.0,8.0,-1.410389,0.755744,-0.045193
0,2,3,7,1914,47.5658,-122.389,1900,5800,455000.0,566500.0,3.0,7.0,-1.246322,-0.308924,-0.235197


In [44]:
pipeline.run_inference_pipeline(df)

waterfront,view,condition,grade,yr_built,lat,long,sqft_living15,sqft_lot15,price,median_price,median_condition,median_grade,PC1,PC2,PC3
i64,i64,i64,i64,i64,f64,f64,i64,i64,f64,f64,f64,f64,f64,f64,f64
0,0,3,8,2007,47.3862,-122.048,3280,4033,429900.0,341000.0,3.0,7.0,2.977914,0.850386,-0.574438
0,0,2,7,1979,47.3035,-122.382,1310,7865,233000.0,269000.0,3.0,8.0,-1.410389,0.755744,-0.045193
0,2,3,7,1914,47.5658,-122.389,1900,5800,455000.0,566500.0,3.0,7.0,-1.246322,-0.308924,-0.235197
0,0,3,8,1985,47.3187,-122.39,1790,7488,258950.0,269000.0,3.0,8.0,-1.340683,-0.067125,0.064354
0,0,4,7,1947,47.6859,-122.395,1690,5962,555000.0,545000.0,3.0,7.0,-1.674613,1.004601,0.057137
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
0,0,3,7,1947,47.7144,-122.319,1000,6947,378000.0,425000.0,3.0,7.0,-2.002895,-0.008679,-0.000545
0,0,3,8,2014,47.2974,-122.349,2927,5183,399950.0,269000.0,3.0,8.0,1.559214,-1.323159,-0.15254
0,0,3,7,2004,47.681,-122.032,1690,2650,575000.0,629495.0,3.0,8.0,0.452837,-1.196911,-0.306607
